In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


This notebook determines the range of fire areas in which the 3 fire area datasets are in agreement by minimizing the Kuiper statistic of their CDFs over the region where the datasets overlap spatially and temporally.

In [6]:
import numpy as np

from wildfire_stats import config as cfg
from wildfire_stats.data import io as dio
from wildfire_stats.plotting import dataset_comp as pdc
from wildfire_stats.reporting import summaries as rsum

P = cfg.load_paths()
A = cfg.load_assumptions()

# MTBS subset: "conus", "ak", or "all" (CONUS + AK; HI/PR always excluded)
MTBS_REGION = A["mtbs_region"]
MTBS_TAG = dio.mtbs_tag(MTBS_REGION)  # output filename prefix, e.g. mtbs_all
MTBS_LABEL = f"MTBS ({MTBS_REGION})"

SAVE_FIGS = True

### MTBS vs. GFD

**MTBS region.** `MTBS_REGION` (set in the setup cell; default `mtbs_region` in `config/default_assumptions.yaml`) selects `conus`, `ak`, or `all` (CONUS + Alaska; Hawai'i and Puerto Rico are always excluded). The Idaho comparisons only overlap CONUS fires, so with `ak` the overlap sets below are empty; that is expected.

In [4]:
mtbs_proj = dio.load_mtbs(P.mtbs / "S_USA.MTBS_BURN_AREA_BOUNDARY.shp",
                          area_col="area_km2", region=MTBS_REGION)
idaho_gdf = dio.load_idaho(P.idaho / "HFD_HistoricFiresDatabase8_0.gdb")

# MTBS starts in 1984; GFA / MODIS cover modis_years
mtbs_proj = dio.filter_years(mtbs_proj, "YEAR", (1984, 2016))
idaho_gdf = dio.filter_years(idaho_gdf, "FIRE_YEAR", (1984, 2016))
idaho_gdf2 = dio.filter_years(idaho_gdf, "FIRE_YEAR", A["modis_years"])

mtbs_overlap = dio.overlap_with(mtbs_proj, idaho_gdf, dedup_col="FIRE_ID")

rsum.print_summary(MTBS_LABEL, mtbs_overlap, "YEAR")
rsum.print_summary("Idaho", idaho_gdf, "FIRE_YEAR")

/Users/ambauer/Research/Projects/2026-1-Pres-Wildfires-BCKKS/wildfire_stats/.env/lib/python3.10/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(
/Users/ambauer/Research/Projects/2026-1-Pres-Wildfires-BCKKS/wildfire_stats/.env/lib/python3.10/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(



MTBS (all) dataset:
  Number of fires: 8736
  Year range: 1984 - 2016
  Mean fire size: 38.23 km²
  Median fire size: 11.94 km²
  Min fire size: 4.05 km²
  Max fire size: 2296.22 km²
  Total burned area: 333992.45 km²

Idaho dataset:
  Number of fires: 33354
  Year range: 1984 - 2016
  Mean fire size: 11.03 km²
  Median fire size: 0.67 km²
  Min fire size: 0.00 km²
  Max fire size: 2462.94 km²
  Total burned area: 367836.11 km²


In [ ]:
alphas = np.linspace(0, 300, 2000)

pdc.plot_cdf_vs_area(mtbs_overlap, idaho_gdf, labels=(MTBS_LABEL, "Idaho"),
                     save_figs=SAVE_FIGS)
pdc.plot_alpha_vs_kuiper_with_min_cdf(mtbs_overlap, idaho_gdf, 
                                      alphas, labels=(MTBS_LABEL, "Idaho"),
                                      save_figs=SAVE_FIGS)

### GFA vs. GFD

In [ ]:
gfa_combined_gdf = dio.load_gfa(P.gfa, verbose=True)

gfa_overlap = dio.overlap_with(gfa_combined_gdf, idaho_gdf2, dedup_col="geometry")

In [ ]:
gfa_combined_gdf

In [ ]:
rsum.print_summary("GFA (overlap)", gfa_overlap, "YEAR")
rsum.print_summary("Idaho", idaho_gdf2, "FIRE_YEAR")

In [ ]:
alphas2 = np.linspace(0, 200, 2000)

pdc.plot_cdf_vs_area(gfa_overlap, idaho_gdf2, labels=("GFA", "Idaho"),
                     save_figs=SAVE_FIGS)
pdc.plot_alpha_vs_kuiper_with_min_cdf(gfa_overlap, idaho_gdf2, alphas2, 
                                      labels=("GFA", "Idaho"),
                                      save_figs=SAVE_FIGS)

### GFA vs. MTBS

In [ ]:
pdc.plot_cdf_vs_area(gfa_overlap, mtbs_overlap, labels=("GFA", MTBS_LABEL),
                     save_figs=SAVE_FIGS)
pdc.plot_alpha_vs_kuiper_with_min_cdf(gfa_overlap, mtbs_overlap, alphas2, 
                                      labels=("GFA", MTBS_LABEL),
                                      save_figs=SAVE_FIGS)

Based on the minimum fire size cutoffs that minimize the Kuiper statistic, all future notebooks will remove fires less than 4 km^2.